In [1]:
#| default_exp module

# 00 · Building blocks

> The small neural-network parts LeWM is assembled from. This notebook is the **source** for `lewm/module.py` — edit here, then run `uv run nbdev-export`.

LeWM is a **world model**: it learns to predict what happens next, not in pixel
space, but in a compact *latent* space of its own devising.

This notebook builds the pieces. The next one (`01_jepa`) wires them together.

## The shape vocabulary

Every tensor in this codebase is named with these letters. Keep them in mind and
the code reads itself:

| Symbol | Meaning | Typical value |
|:---|:---|:---|
| `B` | **B**atch — independent trajectories | 128 |
| `T` | **T**ime — frames in a sequence | 4 |
| `D` | **D**imension of an embedding | 192 |
| `S` | **S**amples — candidate action plans (planning only) | 300 |
| `C, H, W` | Image channels, height, width | 3, 224, 224 |
| `A` | Raw action dimension | 10 |

So `(B, T, D)` is "for each of B trajectories, for each of T timesteps, a
D-number vector". That single shape appears everywhere below.

## Imports

`einops.rearrange` is used throughout instead of `.permute()` / `.view()`. It
names the axes, so `"b t (h d) -> b h t d"` says exactly what moves where —
much harder to get silently wrong than a list of integers.

In [2]:
#| export
import torch
from torch import nn
import torch.nn.functional as F
from einops import rearrange

## `modulate` — how conditioning is injected

The predictor must be told *which action was taken*. A common trick (from
DiT / adaptive layer norm) is to let the conditioning signal rescale and shift
the activations rather than just being concatenated to them.

$$\text{modulate}(x, \text{shift}, \text{scale}) = x \cdot (1 + \text{scale}) + \text{shift}$$

The `1 +` matters: when `scale` and `shift` are zero, this is the identity. The
network starts as if the conditioning were not there and *learns* to use it.
That is the "zero" in AdaLN-zero, and it is a large part of why training is
stable.

In [3]:
#| export
def modulate(x, shift, scale):
    """AdaLN-zero modulation

    x, shift, scale: all broadcastable, typically (B, T, D).
    Returns the same shape as x.

    With shift=scale=0 this returns x unchanged, so a freshly initialised
    block is the identity and conditioning is learned from zero.
    """
    return x * (1 + scale) + shift

## `SIGReg` — the regularizer that prevents collapse

This is **the** idea of the paper, so it is worth slowing down.

### The collapse problem

The training loss asks the model to make its prediction of the next embedding
match the true next embedding. There is a degenerate way to score perfectly:
**map every image to the same constant vector.** Prediction error becomes zero
and the model has learned nothing. This is *representation collapse*, and it is
why most JEPAs need stop-gradients, momentum encoders, or extra loss terms.

### The fix

LeWM adds one term: force the embeddings to look like samples from an
**isotropic Gaussian**. A constant vector is nothing like a Gaussian — it has
zero variance — so collapse becomes impossible by construction.

### How it is measured, cheaply

Testing "is this 192-dimensional cloud Gaussian?" directly is hard. Instead
SIGReg uses a classic trick: a distribution is Gaussian if and only if **every
1-D projection of it is Gaussian**. So:

1. Draw `num_proj` random unit directions.
2. Project the embeddings onto each one, giving 1-D samples.
3. Test each 1-D sample for Gaussianity with the **Epps–Pulley** statistic.

Epps–Pulley compares the *empirical characteristic function* of the data
against the known one for a standard Gaussian, $\varphi(t)=e^{-t^2/2}$:

$$\text{EP} = \int \left| \hat\varphi(t) - e^{-t^2/2} \right|^2 w(t)\,dt$$

The integral is approximated with the trapezoid rule at `knots` points, which
is what the buffers below precompute.

In [4]:
#| export
class SIGReg(torch.nn.Module):
    """Sketch Isotropic Gaussian Regularizer (single-GPU!)

    Penalises embeddings for not looking like an isotropic Gaussian, which is
    what stops the encoder collapsing to a constant.

    Args:
        knots: quadrature points for the Epps-Pulley integral over t in [0, 3].
        num_proj: how many random 1-D directions to test per call. More
            projections means a lower-variance estimate at linear cost.
    """

    def __init__(self, knots=17, num_proj=1024):
        super().__init__()
        self.num_proj = num_proj

        # Quadrature grid over t in [0, 3]: where we sample the characteristic
        # function. Beyond t=3 the Gaussian window below is ~1e-5, so the
        # integrand is negligible and the range can stop there.
        t = torch.linspace(0, 3, knots, dtype=torch.float32)   # (knots,)
        dt = 3 / (knots - 1)

        # Trapezoid rule: interior points get full weight 2*dt, the two
        # endpoints get half that.
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt

        # phi(t) = exp(-t^2/2): the characteristic function of a standard
        # Gaussian, and simultaneously the weighting window w(t).
        window = torch.exp(-t.square() / 2.0)                  # (knots,)

        # Buffers, not parameters: fixed constants that must follow the module
        # across .to(device) but are never trained.
        self.register_buffer("t", t)
        self.register_buffer("phi", window)
        self.register_buffer("weights", weights * window)

    def forward(self, proj):
        """
        proj: (T, B, D)  -- note time first; train.py passes emb.transpose(0, 1)
        returns: scalar
        """
        # 1. Random unit directions: (D, num_proj)
        A = torch.randn(proj.size(-1), self.num_proj, device=proj.device)
        A = A.div_(A.norm(p=2, dim=0))          # normalise each column to length 1

        # 2. Project onto every direction, then evaluate at every knot t.
        #    proj @ A          -> (T, B, num_proj)
        #    .unsqueeze(-1)*t  -> (T, B, num_proj, knots)
        x_t = (proj @ A).unsqueeze(-1) * self.t

        # 3. Empirical characteristic function, averaged over the batch (dim -3):
        #       real part  E[cos(tx)]  should equal phi(t)
        #       imag part  E[sin(tx)]  should equal 0
        #    err is the squared distance between the two, per (T, num_proj, knots).
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()

        # 4. Integrate over t with the trapezoid weights, scale by batch size
        #    (the Epps-Pulley statistic is defined with an n factor).
        statistic = (err @ self.weights) * proj.size(-2)   # (T, num_proj)

        return statistic.mean() # average over projections and time

### Sanity check: does SIGReg actually detect collapse?

The claim above is that Gaussian embeddings score low and collapsed ones score
high. That is testable in three lines — and it is the whole reason the
architecture works, so it is worth verifying rather than trusting.

In [5]:
sig = SIGReg()

# (T, B, D) as forward() expects
gaussian = torch.randn(4, 256, 192)              # healthy: unit Gaussian
collapsed = torch.zeros(4, 256, 192) + 0.5       # collapsed: every vector identical
scaled = torch.randn(4, 256, 192) * 5.0          # wrong scale: Gaussian but too wide

for name, x in [("healthy Gaussian", gaussian),
                ("collapsed constant", collapsed),
                ("scaled x5", scaled)]:
    print(f"{name:20s} SIGReg = {sig(x).item():10.2f}")

healthy Gaussian     SIGReg =       1.05
collapsed constant   SIGReg =     153.72
scaled x5            SIGReg =     215.34


The collapsed input scores orders of magnitude higher. Gradient descent pushes
this number down, so it pushes the encoder *away* from collapse. That is the
entire anti-collapse mechanism — no stop-gradient, no momentum encoder.

Note the third case: embeddings can be perfectly Gaussian in *shape* and still
be penalised for having the wrong scale. SIGReg pins down the distribution, not
just its spread.

## `FeedForward` — the MLP inside a transformer block

Standard transformer plumbing: normalise, expand to a wider hidden dimension,
apply a nonlinearity, project back. The width expansion (`hidden_dim` is
typically 4x `dim`) is where most of a transformer's parameters live.

Shape is unchanged end to end: `(B, T, D) -> (B, T, D)`.

In [6]:
#| export
class FeedForward(nn.Module):
    """FeedForward network used in Transformers

    (B, T, dim) -> (B, T, dim), widening to hidden_dim in the middle.
    """

    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim),   # widen
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),   # back to dim
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## `Attention` — causal self-attention

Each timestep looks at other timesteps and pulls in what is relevant.

**`is_causal=True` is essential here.** It masks out the future, so the
embedding at step `t` can only attend to steps `<= t`. Without it the predictor
could peek at the frame it is supposed to predict and the task would be
trivial.

The shape dance, in one place:

```
x                 (B, T, D)
to_qkv(x)         (B, T, 3 * heads * dim_head)   one matmul for q, k and v
  .chunk(3)       three of (B, T, heads * dim_head)
rearrange         (B, heads, T, dim_head)        split the heads apart
attention         (B, heads, T, dim_head)        each head attends separately
rearrange         (B, T, heads * dim_head)       glue the heads back together
to_out            (B, T, D)
```

In [7]:
#| export
class Attention(nn.Module):
    """Scaled dot-product attention with causal masking"""

    def __init__(self, dim, heads=8, dim_head=64, dropout=0.0):
        super().__init__()
        inner_dim = dim_head * heads

        # With a single head of exactly dim width, the output projection would
        # be a redundant square matmul, so it is skipped.
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads
        self.scale = dim_head**-0.5
        self.dropout = dropout
        self.norm = nn.LayerNorm(dim)
        self.attend = nn.Softmax(dim=-1)

        # One fused projection producing q, k and v together, split below.
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)

        self.to_out = (
            nn.Sequential(nn.Linear(inner_dim, dim), nn.Dropout(dropout))
            if project_out
            else nn.Identity()
        )

    def forward(self, x, causal=True):
        """
        x : (B, T, D)  ->  (B, T, D)

        causal=True masks the future: step t sees only steps <= t. This is what
        keeps the predictor honest -- without it, it could read the answer.
        """
        x = self.norm(x)
        drop = self.dropout if self.training else 0.0

        qkv = self.to_qkv(x).chunk(3, dim=-1)  # q, k, v: (B, heads, T, dim_head)
        q, k, v = (rearrange(t, "b t (h d) -> b h t d", h=self.heads) for t in qkv)

        # Fused kernel: handles the scale, the causal mask and the softmax.
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=drop, is_causal=causal)

        out = rearrange(out, "b h t d -> b t (h d)")
        return self.to_out(out)

## `ConditionalBlock` — a transformer block that listens to the action

This is the block the predictor is built from, and where AdaLN-zero earns its
keep.

A normal transformer block does:

```
x = x + attn(norm(x))
x = x + mlp(norm(x))
```

This one derives **six** vectors from the conditioning signal `c` (the action
embedding) and uses them to steer both sub-layers:

- `shift`, `scale` — modulate the *input* of the sub-layer
- `gate` — scales the sub-layer's *output* before it is added back

The final layer of `adaLN_modulation` is initialised to **exactly zero**, so at
the start of training every `gate` is 0 and each block is the identity function.
The network begins as a no-op and grows into using the conditioning. This is
the single most important initialisation detail in the file.

In [8]:
#| export
class ConditionalBlock(nn.Module):
    """Transformer block with AdaLN-zero conditioning

    forward(x, c): x is the sequence (B, T, D), c is the conditioning signal
    (B, T, D) -- here, the encoded action.
    """

    def __init__(self, dim, heads, dim_head, mlp_dim, dropout=0.0):
        super().__init__()

        self.attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
        self.mlp = FeedForward(dim, mlp_dim, dropout=dropout)

        # elementwise_affine=False: these norms have no learnable scale/bias of
        # their own, because the modulation below supplies exactly that, per
        # example and per timestep.
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)

        # c -> 6 * dim, later chunked into the six modulation vectors.
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(), nn.Linear(dim, 6 * dim, bias=True)
        )

        # The "zero" of AdaLN-zero: start as the identity, learn to condition.
        nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.adaLN_modulation[-1].bias, 0)

    def forward(self, x, c):
        # Six (B, T, D) vectors from one (B, T, 6D) projection.
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
            self.adaLN_modulation(c).chunk(6, dim=-1)
        )
        # gate_* starts at 0, so both residual updates start at zero.
        x = x + gate_msa * self.attn(modulate(self.norm1(x), shift_msa, scale_msa))
        x = x + gate_mlp * self.mlp(modulate(self.norm2(x), shift_mlp, scale_mlp))
        return x

## `Block` — the unconditioned variant

The same thing without conditioning, for stacks that have no action to attend
to. `Transformer` below picks between the two.

In [9]:
#| export
class Block(nn.Module):
    """Standard Transformer block

    (B, T, D) -> (B, T, D). No conditioning: forward takes x only.
    """

    def __init__(self, dim, heads, dim_head, mlp_dim, dropout=0.0):
        super().__init__()

        self.attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
        self.mlp = FeedForward(dim, mlp_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

## `Transformer` — the stack

Projects into the working width, runs `depth` blocks, normalises, projects out.

The three `nn.Linear`-or-`nn.Identity` choices are a small efficiency habit: if
the input and hidden widths already match, no projection is inserted at all. In
this repo `input_dim == hidden_dim == output_dim == 192`, so all three are in
fact `Identity` and the stack is pure blocks.

`forward` dispatches on block type, so the same class serves both the plain and
the conditioned stack.

In [10]:
#| export
class Transformer(nn.Module):
    """Standard Transformer with support for AdaLN-zero blocks

    (B, T, input_dim) -> (B, T, output_dim)
    """

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        depth,
        heads,
        dim_head,
        mlp_dim,
        dropout=0.0,
        block_class=Block,
    ):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        self.layers = nn.ModuleList([])

        # Identity when the widths already agree -- no wasted matmul.
        self.input_proj = (
            nn.Linear(input_dim, hidden_dim)
            if input_dim != hidden_dim
            else nn.Identity()
        )

        self.cond_proj = (
            nn.Linear(input_dim, hidden_dim)
            if input_dim != hidden_dim
            else nn.Identity()
        )

        self.output_proj = (
            nn.Linear(hidden_dim, output_dim)
            if hidden_dim != output_dim
            else nn.Identity()
        )

        for _ in range(depth):
            self.layers.append(
                block_class(hidden_dim, heads, dim_head, mlp_dim, dropout)
            )

    def forward(self, x, c=None):

        if hasattr(self, "input_proj"):
            x = self.input_proj(x)

        if c is not None and hasattr(self, "cond_proj"):
            c = self.cond_proj(c)

        # Plain Blocks take x only; ConditionalBlocks also take c.
        for block in self.layers:
            x = block(x) if isinstance(block, Block) else block(x, c)
        x = self.norm(x)

        if hasattr(self, "output_proj"):
            x = self.output_proj(x)
        return x

## `Embedder` — turning raw actions into vectors

An action in PushT is a couple of numbers (where to push). The predictor works
in 192 dimensions, so actions must be lifted to that width.

The `Conv1d` with `kernel_size=1` is the subtle part. A 1x1 convolution over
the time axis is *exactly* a per-timestep linear layer — it mixes the action's
own components but never mixes across time. It is a `Linear` that happens to
want channels-first, which is why the two `permute` calls surround it.

```
x            (B, T, A)
permute      (B, A, T)     channels-first for Conv1d
patch_embed  (B, smoothed_dim, T)
permute      (B, T, smoothed_dim)
embed        (B, T, emb_dim)
```

In [11]:
#| export
class Embedder(nn.Module):
    """Lift raw actions (B, T, A) into embeddings (B, T, emb_dim).

    Args:
        input_dim: raw action width A. In this repo it is
            frameskip * action_dim (5 * 2 = 10 for PushT), because each step
            of the dataset stacks the actions of 5 simulator frames.
        smoothed_dim: width after the 1x1 conv.
        emb_dim: final embedding width, matched to the predictor (192).
        mlp_scale: hidden-width multiplier inside the MLP.
    """

    def __init__(
        self,
        input_dim=10,
        smoothed_dim=10,
        emb_dim=10,
        mlp_scale=4,
    ):
        super().__init__()
        # kernel_size=1 over time == a per-timestep linear map. No mixing
        # across timesteps happens here.
        self.patch_embed = nn.Conv1d(input_dim, smoothed_dim, kernel_size=1, stride=1)
        self.embed = nn.Sequential(
            nn.Linear(smoothed_dim, mlp_scale * emb_dim),
            nn.SiLU(),
            nn.Linear(mlp_scale * emb_dim, emb_dim),
        )

    def forward(self, x):
        """
        x: (B, T, D)
        """
        x = x.float()
        x = x.permute(0, 2, 1)     # (B, T, A) -> (B, A, T) for Conv1d
        x = self.patch_embed(x)    # (B, smoothed_dim, T)
        x = x.permute(0, 2, 1)     # back to (B, T, smoothed_dim)
        x = self.embed(x)          # (B, T, emb_dim)
        return x

## `MLP` — the projector head

Used twice in LeWM: once after the encoder (`projector`) and once after the
predictor (`pred_proj`). Both map 192 -> 2048 -> 192.

Note the default `norm_fn` here is `LayerNorm`, but the Hydra config overrides
it with `BatchNorm1d`. That matters and is not cosmetic: BatchNorm normalises
*across the batch*, which actively fights collapse by making it impossible for
every example to take the same value without the normaliser blowing up. It
works with SIGReg rather than duplicating it.

BatchNorm is also why this operates on flattened `(B*T, D)` input — it needs a
real batch axis, so `lewm/jepa.py` folds time into the batch before calling it.

In [12]:
#| export
class MLP(nn.Module):
    """Simple MLP with optional normalization and activation

    (N, input_dim) -> (N, output_dim), where N is typically B*T.
    """

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim=None,
        norm_fn=nn.LayerNorm,
        act_fn=nn.GELU,
    ):
        super().__init__()
        # The config passes BatchNorm1d here, which needs a flat batch axis --
        # hence the (B*T, D) input convention.
        norm_fn = norm_fn(hidden_dim) if norm_fn is not None else nn.Identity()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            norm_fn,
            act_fn(),
            nn.Linear(hidden_dim, output_dim or input_dim),
        )

    def forward(self, x):
        """
        x: (B*T, D)
        """
        return self.net(x)

## `ARPredictor` — the world model's dynamics

The piece that answers *"given where I am and what I do, where do I end up?"*

It is a causal transformer over the embedding sequence, conditioned on actions:

- `pos_embedding` is a learned marker telling the model which timestep is which
  (attention alone is order-blind).
- `ConditionalBlock` stacks inject the action at every layer.
- Causal masking means output `t` depends only on inputs `<= t`.

That last property is what makes autoregressive rollout valid: the model can be
fed its own predictions and asked for one more step, over and over.

In [13]:
#| export
class ARPredictor(nn.Module):
    """Autoregressive predictor for next-step embedding prediction.

    forward(x, c): x is (B, T, input_dim) embeddings, c is (B, T, input_dim)
    action embeddings. Returns (B, T, output_dim): the predicted *next*
    embedding for each timestep.

    Args:
        num_frames: longest sequence the position embedding supports. Equals
            history_size (3) in this repo.
    """

    def __init__(
        self,
        *,
        num_frames,
        depth,
        heads,
        mlp_dim,
        input_dim,
        hidden_dim,
        output_dim=None,
        dim_head=64,
        dropout=0.0,
        emb_dropout=0.0,
    ):
        super().__init__()
        # Learned position codes: attention is permutation-invariant without them.
        self.pos_embedding = nn.Parameter(torch.randn(1, num_frames, input_dim))
        self.dropout = nn.Dropout(emb_dropout)
        self.transformer = Transformer(
            input_dim,
            hidden_dim,
            output_dim or input_dim,
            depth,
            heads,
            dim_head,
            mlp_dim,
            dropout,
            block_class=ConditionalBlock,   # conditioned on the action
        )

    def forward(self, x, c):
        """
        x: (B, T, d)
        c: (B, T, act_dim)
        """
        T = x.size(1)
        x = x + self.pos_embedding[:, :T]   # slice: sequences may be shorter
        x = self.dropout(x)
        x = self.transformer(x, c)
        return x

### Check the predictor runs and respects causality

Two things worth proving rather than assuming: the shapes line up, and the
causal mask genuinely works. The second test perturbs *only the last timestep*
of the input — if the mask is correct, earlier outputs must not move at all.

In [14]:
pred = ARPredictor(
    num_frames=3, depth=2, heads=4, mlp_dim=256,
    input_dim=192, hidden_dim=192, output_dim=192,
).eval()

B, T, D = 2, 3, 192
emb = torch.randn(B, T, D)
act = torch.randn(B, T, D)

with torch.no_grad():
    out = pred(emb, act)
print(f"in  {tuple(emb.shape)} + action {tuple(act.shape)}")
print(f"out {tuple(out.shape)}")

# Causality: change the LAST timestep only; earlier outputs must be identical.
emb2 = emb.clone()
emb2[:, -1] += 10.0
with torch.no_grad():
    out2 = pred(emb2, act)

delta = (out2 - out).abs().amax(dim=(0, 2))    # largest change at each timestep
print(f"\nmax output change per timestep: {[f'{d:.2e}' for d in delta]}")
print("timesteps 0..T-2 unchanged ->", torch.allclose(out2[:, :-1], out[:, :-1], atol=1e-5))

in  (2, 3, 192) + action (2, 3, 192)
out (2, 3, 192)

max output change per timestep: ['0.00e+00', '0.00e+00', '8.34e-07']
timesteps 0..T-2 unchanged -> True


The first `T-1` outputs are bit-identical; only the last moved. The model cannot
see the future, so feeding it its own predictions during rollout is sound.

## Export

`uv run nbdev-export` reads the `#| export` markers above and writes `lewm/module.py`.
Run it from the repo root after editing this notebook:

```bash
uv run nbdev-export
```

In [15]:
# Export explicitly from the repository root: uv run nbdev-export